In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 7


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.480626977980137
Epoch 2/100, Loss: 2.223680555820465
Epoch 3/100, Loss: 2.0442496240139008
Epoch 4/100, Loss: 2.3881320729851723
Epoch 5/100, Loss: 2.3346633166074753
Epoch 6/100, Loss: 2.371195562183857
Epoch 7/100, Loss: 2.321855142712593
Epoch 8/100, Loss: 2.197446748614311
Epoch 9/100, Loss: 2.283734455704689
Epoch 10/100, Loss: 2.229678638279438
Epoch 11/100, Loss: 2.380580112338066
Epoch 12/100, Loss: 2.2964107170701027
Epoch 13/100, Loss: 2.3758918419480324


Epoch 14/100, Loss: 2.215927168726921
Epoch 15/100, Loss: 2.3479418605566025
Epoch 16/100, Loss: 2.2494197711348534
Epoch 17/100, Loss: 2.30117604136467
Epoch 18/100, Loss: 2.1517966762185097
Epoch 19/100, Loss: 2.113001599907875
Epoch 20/100, Loss: 2.430112846195698
Epoch 21/100, Loss: 2.308952644467354
Epoch 22/100, Loss: 2.313068211078644
Epoch 23/100, Loss: 2.205165594816208
Epoch 24/100, Loss: 2.4303023889660835
Epoch 25/100, Loss: 3.0137993693351746
Epoch 26/100, Loss: 2.476198688149452
Epoch 27/100, Loss: 2.105695754289627
Epoch 28/100, Loss: 2.2678909078240395
Epoch 29/100, Loss: 2.2178980857133865


Epoch 30/100, Loss: 2.1127709299325943
Epoch 31/100, Loss: 2.2315320298075676
Epoch 32/100, Loss: 2.197834827005863
Epoch 33/100, Loss: 2.213390029966831
Epoch 34/100, Loss: 2.235645644366741
Epoch 35/100, Loss: 2.11887289583683
Epoch 36/100, Loss: 2.4171052053570747
Epoch 37/100, Loss: 2.252612166106701
Epoch 38/100, Loss: 2.109278604388237
Epoch 39/100, Loss: 2.3840047419071198
Epoch 40/100, Loss: 2.2904757037758827
Epoch 41/100, Loss: 2.4502880722284317
Epoch 42/100, Loss: 2.2359855249524117
Epoch 43/100, Loss: 2.374554380774498
Epoch 44/100, Loss: 2.2537521943449974
Epoch 45/100, Loss: 2.500761851668358
Epoch 46/100, Loss: 2.240357778966427


Epoch 47/100, Loss: 2.2919858396053314
Epoch 48/100, Loss: 2.206377513706684
Epoch 49/100, Loss: 2.052030049264431
Epoch 50/100, Loss: 2.332518197596073
Epoch 51/100, Loss: 2.3589997738599777
Epoch 52/100, Loss: 2.5547488257288933
Epoch 53/100, Loss: 2.1542973071336746
Epoch 54/100, Loss: 2.129869431257248
Epoch 55/100, Loss: 2.2552609518170357
Epoch 56/100, Loss: 2.383612923324108
Epoch 57/100, Loss: 2.4064295142889023
Epoch 58/100, Loss: 2.341582752764225
Epoch 59/100, Loss: 2.2279864475131035
Epoch 60/100, Loss: 2.2628786638379097
Epoch 61/100, Loss: 1.981223288923502
Epoch 62/100, Loss: 2.2104636430740356
Epoch 63/100, Loss: 2.321546792984009
Epoch 64/100, Loss: 2.2818131744861603


Epoch 65/100, Loss: 2.3855917155742645
Epoch 66/100, Loss: 2.1924814954400063
Epoch 67/100, Loss: 2.3117478638887405
Epoch 68/100, Loss: 2.168741226196289
Epoch 69/100, Loss: 2.438064582645893
Epoch 70/100, Loss: 2.2127014100551605
Epoch 71/100, Loss: 2.1021128594875336
Epoch 72/100, Loss: 2.1407655999064445
Epoch 73/100, Loss: 2.230172999203205
Epoch 74/100, Loss: 2.3308120146393776
Epoch 75/100, Loss: 2.2291010320186615
Epoch 76/100, Loss: 2.393808327615261
Epoch 77/100, Loss: 2.160465721040964
Epoch 78/100, Loss: 2.2913262844085693
Epoch 79/100, Loss: 2.7260944098234177
Epoch 80/100, Loss: 2.3697289675474167
Epoch 81/100, Loss: 2.6936553567647934


...

Epoch 34/100, Loss: 2.0995666161179543
Epoch 35/100, Loss: 2.1212755143642426
Epoch 36/100, Loss: 2.1546171233057976
Epoch 37/100, Loss: 2.0825662910938263
Epoch 38/100, Loss: 2.1053971722722054
Epoch 39/100, Loss: 2.1421447321772575
Epoch 40/100, Loss: 2.1625735983252525
Epoch 41/100, Loss: 2.032958433032036
Epoch 42/100, Loss: 2.060432218015194
Epoch 43/100, Loss: 2.1222429051995277
Epoch 44/100, Loss: 2.054679714143276
Epoch 45/100, Loss: 1.9913929626345634


Epoch 46/100, Loss: 2.1458038613200188
Epoch 47/100, Loss: 2.061771832406521
Epoch 48/100, Loss: 2.219875618815422
Epoch 49/100, Loss: 2.1733668446540833
Epoch 50/100, Loss: 1.9971708357334137
Epoch 51/100, Loss: 2.191361904144287
Epoch 52/100, Loss: 2.043178290128708
Epoch 53/100, Loss: 1.9649239033460617
Epoch 54/100, Loss: 2.162031002342701
Epoch 55/100, Loss: 2.1539072394371033
Epoch 56/100, Loss: 2.0973666310310364
Epoch 57/100, Loss: 2.124430112540722


Epoch 58/100, Loss: 2.0192531794309616
Epoch 59/100, Loss: 2.0074498876929283
Epoch 60/100, Loss: 2.0608309507369995
Epoch 61/100, Loss: 2.177150011062622
Epoch 62/100, Loss: 2.071597643196583
Epoch 63/100, Loss: 2.052926354110241
Epoch 64/100, Loss: 2.1433739364147186
Epoch 65/100, Loss: 2.114037111401558
Epoch 66/100, Loss: 2.1341497749090195
Epoch 67/100, Loss: 2.0690767094492912
Epoch 68/100, Loss: 2.098940595984459
Epoch 69/100, Loss: 2.044538915157318
Epoch 70/100, Loss: 2.1797594279050827
Epoch 71/100, Loss: 2.1171073243021965
Epoch 72/100, Loss: 2.0816774740815163
Epoch 73/100, Loss: 2.3203962296247482
Epoch 74/100, Loss: 2.097874879837036


Epoch 75/100, Loss: 2.1062845960259438
Epoch 76/100, Loss: 1.9830825552344322
Epoch 77/100, Loss: 2.040193483233452
Epoch 78/100, Loss: 2.1362826600670815
Epoch 79/100, Loss: 2.1297972425818443
Epoch 80/100, Loss: 2.1507687717676163
Epoch 81/100, Loss: 2.054323695600033
Epoch 82/100, Loss: 2.2144846841692924
Epoch 83/100, Loss: 2.0170069336891174
Epoch 84/100, Loss: 2.079614944756031
Epoch 85/100, Loss: 2.219583176076412
Epoch 86/100, Loss: 2.2273652106523514
Epoch 87/100, Loss: 2.1799968257546425
Epoch 88/100, Loss: 2.1206018030643463
Epoch 89/100, Loss: 2.1514230966567993


Epoch 90/100, Loss: 2.164983779191971
Epoch 91/100, Loss: 2.175075612962246
Epoch 92/100, Loss: 2.0341009199619293
Epoch 93/100, Loss: 2.055366925895214
Epoch 94/100, Loss: 2.1180111467838287
Epoch 95/100, Loss: 2.197652667760849
Epoch 96/100, Loss: 2.0567188188433647
Epoch 97/100, Loss: 2.1588582769036293
Epoch 98/100, Loss: 2.0918185487389565
Epoch 99/100, Loss: 2.131632551550865
Epoch 100/100, Loss: 2.0320114344358444
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5459
